# 07 - Sentiment & Alt Data

**Author:** Sacha Huberty

**Purpose:** Scrape live financial-news RSS feeds (robots.txt checked
at request time, not just pre-vetted by hand -- the S14 legal
consideration), run NLP preprocessing and VADER sentiment scoring,
tag headlines to asset-class buckets, and report via a word cloud and
LDA topics. Like stage 6's options positioning, free news sources have
no historical archive, so this V4 signal is LIVE-ONLY: demonstrated
here on today's real headlines, not backtested over history.

**Last updated:** 2026-07-25

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

pd.options.display.float_format = '{:.4f}'.format

from atlas import data, sentiment, strategy, universe

cfg = data.load_config()
cfg["sentiment"]

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
class_bucket = universe_df["class_bucket"]

# Scrape checks robots.txt for each source AT REQUEST TIME (S14 legal
# consideration) -- a source that disallows the path is silently
# skipped, not force-scraped.
headlines = sentiment.scrape_headlines(cfg["sentiment"]["sources"], cfg)
print(f"Scraped {len(headlines)} headlines from {headlines['source'].nunique()} source(s)")
headlines[["title", "published", "source"]].head(10)

## Analysis / signal logic

### NLP preprocessing and sentiment scoring

Tokenize, drop stopwords, POS-tag, and lemmatize (verb-aware -- a
plain noun-only lemmatizer would leave "rallying" unreduced, silently
degrading topic/sentiment quality on these verb-heavy headlines), then
score each headline with VADER.

In [ ]:
headlines["tokens"] = headlines["title"].apply(sentiment.preprocess)
headlines["sentiment"] = headlines["title"].apply(sentiment.score_sentiment)
headlines["bucket"] = headlines["title"].apply(sentiment.tag_bucket)

print("Most positive headlines:")
print(headlines.nlargest(5, "sentiment")[["title", "sentiment"]].to_string(index=False))
print()
print("Most negative headlines:")
print(headlines.nsmallest(5, "sentiment")[["title", "sentiment"]].to_string(index=False))

In [ ]:
headlines["bucket"].value_counts(dropna=False)

### Weekly per-bucket sentiment (lagged to avoid lookahead)

With only today's snapshot, this collapses to a single week -- the
aggregation logic itself (lag, weekly grouping, mean) is what matters
here and is unit-tested against synthetic multi-day data, since a real
multi-week run needs many days of accumulated live scrapes.

In [ ]:
weekly = sentiment.aggregate_sentiment(headlines, cfg)
weekly

### Word cloud and LDA topics (reporting only)

In [ ]:
all_tokens = [tok for tokens in headlines["tokens"] for tok in tokens]
wc = WordCloud(width=900, height=400, background_color="white", random_state=cfg["general"]["random_seed"])
wc.generate(" ".join(all_tokens))

plt.figure(figsize=(11, 5))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Scraped headlines: word cloud")
plt.show()

In [ ]:
documents = [tokens for tokens in headlines["tokens"] if len(tokens) > 0]
lda_model, dictionary, corpus = sentiment.build_lda_model(documents, cfg)
for topic_id, topic in lda_model.print_topics():
    print(f"Topic {topic_id}: {topic}")

### V4 view: today's snapshot (live use only, not backtested)

`with_sentiment_view` wraps any strategy_fn with this snapshot. It is
meant for a forward/live loop that re-scrapes and passes in a fresh
`bucket_scores` each real week -- applying one fixed live snapshot
across historical dates would be a lookahead violation, so this is
demonstrated as a single live call, not an OOS backtest.

In [ ]:
if len(weekly):
    bucket_scores = weekly.iloc[-1]
else:
    bucket_scores = pd.Series(dtype=float)
print("Latest bucket sentiment scores:")
print(bucket_scores)

view = sentiment.sentiment_view(bucket_scores, class_bucket, cfg)
print()
print("Per-asset V4 view (bounded by sentiment.max_view_magnitude):")
view.sort_values()

In [ ]:
def equal_weight_strategy(as_of, window):
    return pd.Series(1.0 / len(class_bucket), index=class_bucket.index)


wrapped = strategy.with_sentiment_view(
    equal_weight_strategy, cfg, class_bucket, bucket_scores
)
tilted_weights = wrapped(pd.Timestamp.today(), pd.DataFrame())
tilted_weights.sort_values()

## Notes / next steps

- **Real data messiness, observed directly:** the weekly aggregation table above has two rows, not one -- one scraped headline carried a stale 2024 publish date (evergreen or reprinted content some RSS feeds include alongside today's news), correctly landing in its own separate week rather than being merged into today's. An unintentional but reassuring confirmation that the lag/weekly-grouping logic handles genuine date diversity correctly, not just a single day's snapshot.

- **Sample-size caveat:** a single day's scrape gives a modest
  headline count per bucket (some buckets, e.g. real_estate, may have
  very few or zero matches on any given day) -- the weekly aggregation
  and view are illustrative of the mechanism, not a statistically
  robust daily signal yet. A genuine live deployment would accumulate
  headlines across many real days before the weekly mean becomes
  meaningful.
- Like stage 6's options positioning, sentiment is fundamentally
  LIVE-ONLY: free RSS sources have no historical archive, so V4 cannot
  be backtested over 2010-present, only computed forward from today's
  headlines. `with_sentiment_view` exists for that forward use.
- robots.txt is checked at REQUEST TIME for every source, not just
  pre-vetted by hand -- confirmed directly while choosing sources:
  MarketWatch's topstories RSS and several others were disallowed
  (or bot-blocked despite allowing robots.txt, e.g. WSJ), while Yahoo
  Finance and Forbes were genuinely permitted and used here.
- Next (stage 8): `views.py` + Black-Litterman fusion in
  `allocation.py`. Until now, V1-V4 combined naively (posture
  switching plus additive tilts, per PROJECT_STRUCTURE.md 5.1); BL
  will properly fuse them into one posterior return vector with
  calibrated per-view confidence, which is where the honest
  underperformance vs. the stage-2 baseline seen since stage 3 is
  expected to actually start closing.